# classifier3

목표: `cf2` 폴더의 객체 검출 JSON을 다시 읽어서, 각 프레임/구간별로  
헬멧 미착용, 인도 주행 등 C파트 자연어 질의에 필요한 이상행동 판정 JSON을 새로 만든다.

이 노트북은 기존 셀을 재사용하지 않고, 다음 흐름만 수행한다.

1. 경로 설정
2. cf2 JSON 로드
3. 객체 구조 정규화
4. 사람-이륜 이동수단 매칭
5. 헬멧 미착용 판정
6. 인도 주행 판정
7. 프레임별/구간별 JSON 저장

In [1]:
# Cell 1. 기본 import

import os
import json
import math
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
from tqdm import tqdm

In [2]:
# Cell 2. 경로 설정

# classifier3.ipynb를 프로젝트 루트에서 실행한다고 가정
ROOT_DIR = Path(".").resolve()

# 기존 객체 분리 JSON 폴더
CF2_DIR = ROOT_DIR / "cf2"

# classifier3 결과 저장 폴더
OUT_DIR = ROOT_DIR / "classifier3_output"
OUT_DIR.mkdir(exist_ok=True)

# 결과 파일
FRAME_RESULT_PATH = OUT_DIR / "frame_anomaly_results.json"
SEGMENT_RESULT_PATH = OUT_DIR / "segment_anomaly_results.json"

print("ROOT_DIR:", ROOT_DIR)
print("CF2_DIR exists:", CF2_DIR.exists(), CF2_DIR)
print("OUT_DIR:", OUT_DIR)

ROOT_DIR: C:\Users\우성한
CF2_DIR exists: True C:\Users\우성한\cf2
OUT_DIR: C:\Users\우성한\classifier3_output


In [3]:
# Cell 3. cf2 JSON 파일 목록 확인

json_files = sorted(CF2_DIR.glob("*.json"))

print("json files:", len(json_files))
for p in json_files[:10]:
    print("-", p.name)

if len(json_files) == 0:
    raise FileNotFoundError(f"cf2 폴더에 json 파일이 없습니다: {CF2_DIR}")

json files: 10
- bs_e4387_c1.json
- bs_e4387_c2.json
- es_e3768_c1.json
- es_e3768_c2.json
- md_e2122_c1.json
- md_e2122_c2.json
- mp_e2730_c1.json
- mp_e2730_c2.json
- ms_e3500_c1.json
- ms_e3500_c2.json


In [4]:
# Cell 4. JSON 로드 함수

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


sample = load_json(json_files[0])
print("sample file:", json_files[0].name)
print("top keys:", sample.keys())

if "frames" in sample and len(sample["frames"]) > 0:
    print("frame keys:", sample["frames"][0].keys())
    if "objects" in sample["frames"][0] and len(sample["frames"][0]["objects"]) > 0:
        print("object example:", sample["frames"][0]["objects"][0])

sample file: bs_e4387_c1.json
top keys: dict_keys(['video_name', 'summary', 'frames'])
frame keys: dict_keys(['frame_id', 'object_count', 'classes', 'objects'])


In [5]:
# Cell 5. bbox 유틸 함수

def bbox_xyxy(obj):
    '''
    obj["bbox"] = [x1, y1, x2, y2] 라고 가정.
    혹시 값이 문자열/float이어도 처리.
    '''
    b = obj.get("bbox", None)
    if b is None or len(b) != 4:
        return None
    x1, y1, x2, y2 = map(float, b)
    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1
    return [x1, y1, x2, y2]


def bbox_center(b):
    x1, y1, x2, y2 = b
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def bbox_area(b):
    x1, y1, x2, y2 = b
    return max(0, x2 - x1) * max(0, y2 - y1)


def iou(b1, b2):
    ax1, ay1, ax2, ay2 = b1
    bx1, by1, bx2, by2 = b2

    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)

    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih

    union = bbox_area(b1) + bbox_area(b2) - inter
    return inter / union if union > 0 else 0


def point_in_bbox(point, b):
    x, y = point
    x1, y1, x2, y2 = b
    return x1 <= x <= x2 and y1 <= y <= y2


def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)

In [6]:
# Cell 6. 객체 정규화

TWO_WHEEL_CLASSES = {"bicycle", "motorcycle", "bike", "scooter"}
PERSON_CLASSES = {"person"}
HELMET_CLASSES = {"helmet"}
NO_HELMET_CLASSES = {"no_helmet", "no-helmet", "without_helmet", "bare_head"}


def normalize_object(obj, idx):
    cls = str(obj.get("class", obj.get("label", ""))).lower()
    conf = float(obj.get("confidence", obj.get("score", 0.0)))
    b = bbox_xyxy(obj)

    return {
        "object_id": obj.get("id", idx),
        "class": cls,
        "confidence": conf,
        "bbox": b,
        "raw": obj,
    }


def normalize_frame(frame):
    frame_id = frame.get("frame_id", frame.get("frame_index", None))
    objects = frame.get("objects", [])

    norm_objects = []
    for idx, obj in enumerate(objects):
        n = normalize_object(obj, idx)
        if n["bbox"] is not None:
            norm_objects.append(n)

    return {
        "frame_id": frame_id,
        "objects": norm_objects,
        "raw": frame,
    }

In [7]:
# Cell 7. 사람-이륜 이동수단 매칭

def bottom_center(b):
    x1, y1, x2, y2 = b
    return ((x1 + x2) / 2, y2)


def person_vehicle_relation_score(person_bbox, vehicle_bbox):
    '''
    사람이 이륜 이동수단을 타고 있는지 추정하는 점수.
    기본 객체 JSON만 가지고 판단하므로 완전한 pose 추정은 아님.
    '''
    px, py = bbox_center(person_bbox)
    vx, vy = bbox_center(vehicle_bbox)

    p_area = bbox_area(person_bbox)
    v_area = bbox_area(vehicle_bbox)

    if p_area <= 0 or v_area <= 0:
        return 0.0

    # 1) bbox가 어느 정도 겹치면 탑승 가능성 증가
    overlap = iou(person_bbox, vehicle_bbox)

    # 2) 사람 중심이 차량 중심보다 위쪽에 있으면 탑승 가능성 증가
    vertical_ok = 1.0 if py < vy else 0.3

    # 3) 수평 중심이 가까우면 탑승 가능성 증가
    _, _, vx2, _ = vehicle_bbox
    vx1, _, _, _ = vehicle_bbox
    vehicle_w = max(1, vx2 - vx1)
    horizontal_dist = abs(px - vx)
    horizontal_score = max(0.0, 1.0 - horizontal_dist / (vehicle_w * 1.2))

    # 4) 사람 발점이 차량 bbox 내부/근처이면 탑승 가능성 증가
    foot = bottom_center(person_bbox)
    foot_inside = 1.0 if point_in_bbox(foot, vehicle_bbox) else 0.0

    score = (
        0.35 * min(overlap / 0.15, 1.0) +
        0.25 * vertical_ok +
        0.25 * horizontal_score +
        0.15 * foot_inside
    )

    return round(float(score), 4)


def match_riders(objects, threshold=0.35):
    persons = [o for o in objects if o["class"] in PERSON_CLASSES]
    vehicles = [o for o in objects if o["class"] in TWO_WHEEL_CLASSES]

    matches = []

    for v in vehicles:
        best = None
        best_score = 0.0

        for p in persons:
            score = person_vehicle_relation_score(p["bbox"], v["bbox"])
            if score > best_score:
                best_score = score
                best = p

        if best is not None and best_score >= threshold:
            matches.append({
                "vehicle": v,
                "rider": best,
                "rider_score": best_score,
            })

    return matches

In [8]:
# Cell 8. 헬멧 판정

def head_region_from_person(person_bbox):
    '''
    사람 bbox의 상단 30%를 머리 영역으로 간주.
    helmet/no_helmet 객체가 별도로 검출되어 있을 때 이 영역과 겹치는지 확인한다.
    '''
    x1, y1, x2, y2 = person_bbox
    h = y2 - y1
    return [x1, y1, x2, y1 + h * 0.30]


def judge_helmet_for_rider(rider_obj, objects):
    '''
    cf2 JSON 안에 helmet 또는 no_helmet 계열 객체가 있으면 활용.
    없다면 unknown으로 남긴다.

    중요:
    기본 YOLO 결과에 person/bicycle/motorcycle만 있으면 헬멧 착용 여부는 이 함수만으로 확정 불가.
    그 경우 helmet_status = "unknown"으로 저장하고, C파트에서 "판정불가"로 검색 가능하게 한다.
    '''
    person_bbox = rider_obj["bbox"]
    head_bbox = head_region_from_person(person_bbox)

    helmets = [o for o in objects if o["class"] in HELMET_CLASSES]
    no_helmets = [o for o in objects if o["class"] in NO_HELMET_CLASSES]

    helmet_hits = []
    for h in helmets:
        if iou(head_bbox, h["bbox"]) > 0.01 or point_in_bbox(bbox_center(h["bbox"]), head_bbox):
            helmet_hits.append(h)

    no_helmet_hits = []
    for nh in no_helmets:
        if iou(head_bbox, nh["bbox"]) > 0.01 or point_in_bbox(bbox_center(nh["bbox"]), head_bbox):
            no_helmet_hits.append(nh)

    if len(no_helmet_hits) > 0:
        return {
            "helmet_status": "no_helmet",
            "helmet_confidence": max(o["confidence"] for o in no_helmet_hits),
            "evidence": "rider head region overlaps no_helmet detection",
            "head_bbox": head_bbox,
        }

    if len(helmet_hits) > 0:
        return {
            "helmet_status": "helmet",
            "helmet_confidence": max(o["confidence"] for o in helmet_hits),
            "evidence": "rider head region overlaps helmet detection",
            "head_bbox": head_bbox,
        }

    return {
        "helmet_status": "unknown",
        "helmet_confidence": 0.0,
        "evidence": "helmet/no_helmet class not found near rider head region",
        "head_bbox": head_bbox,
    }

In [9]:
# Cell 9. 인도 영역 설정 로드

# 영상별 인도 영역 polygon을 저장할 파일.
# 처음에는 없어도 실행 가능하며, 없으면 sidewalk_status는 unknown이 된다.
#
# 예시:
# {
#   "video1.mp4": [
#     [[0, 700], [1920, 650], [1920, 1080], [0, 1080]]
#   ],
#   "video2.mp4": [
#     [[100, 500], [800, 500], [900, 1080], [0, 1080]]
#   ]
# }

SIDEWALK_CONFIG_PATH = ROOT_DIR / "sidewalk_zones.json"

if SIDEWALK_CONFIG_PATH.exists():
    with open(SIDEWALK_CONFIG_PATH, "r", encoding="utf-8") as f:
        SIDEWALK_ZONES = json.load(f)
else:
    SIDEWALK_ZONES = {}

print("sidewalk config exists:", SIDEWALK_CONFIG_PATH.exists())
print("videos with sidewalk zones:", len(SIDEWALK_ZONES))

sidewalk config exists: False
videos with sidewalk zones: 0


In [10]:
# Cell 10. polygon 내부 판정

def point_in_polygon(point, polygon):
    '''
    ray casting algorithm
    point: (x, y)
    polygon: [[x1,y1], [x2,y2], ...]
    '''
    x, y = point
    inside = False
    n = len(polygon)

    if n < 3:
        return False

    p1x, p1y = polygon[0]
    for i in range(n + 1):
        p2x, p2y = polygon[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y + 1e-9) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        p1x, p1y = p2x, p2y

    return inside


def judge_sidewalk_for_vehicle(video_name, vehicle_obj):
    zones = SIDEWALK_ZONES.get(video_name, [])

    vehicle_bbox = vehicle_obj["bbox"]
    vehicle_foot = bottom_center(vehicle_bbox)

    if len(zones) == 0:
        return {
            "sidewalk_status": "unknown",
            "sidewalk_confidence": 0.0,
            "evidence": "sidewalk polygon not configured for this video",
            "vehicle_bottom_center": vehicle_foot,
        }

    for polygon in zones:
        if point_in_polygon(vehicle_foot, polygon):
            return {
                "sidewalk_status": "on_sidewalk",
                "sidewalk_confidence": 1.0,
                "evidence": "vehicle bottom center is inside configured sidewalk polygon",
                "vehicle_bottom_center": vehicle_foot,
            }

    return {
        "sidewalk_status": "not_sidewalk",
        "sidewalk_confidence": 1.0,
        "evidence": "vehicle bottom center is outside configured sidewalk polygon",
        "vehicle_bottom_center": vehicle_foot,
    }

In [11]:
# Cell 11. 프레임 단위 이상행동 판정

def judge_frame(video_name, frame):
    norm = normalize_frame(frame)
    objects = norm["objects"]

    rider_matches = match_riders(objects)

    events = []

    for m in rider_matches:
        vehicle = m["vehicle"]
        rider = m["rider"]

        helmet_result = judge_helmet_for_rider(rider, objects)
        sidewalk_result = judge_sidewalk_for_vehicle(video_name, vehicle)

        anomaly_types = []

        if helmet_result["helmet_status"] == "no_helmet":
            anomaly_types.append("helmet_missing")

        if sidewalk_result["sidewalk_status"] == "on_sidewalk":
            anomaly_types.append("sidewalk_riding")

        # 판정불가도 검색 가능하도록 별도 플래그로 남김
        uncertain_flags = []
        if helmet_result["helmet_status"] == "unknown":
            uncertain_flags.append("helmet_unknown")
        if sidewalk_result["sidewalk_status"] == "unknown":
            uncertain_flags.append("sidewalk_unknown")

        event = {
            "event_type": "two_wheel_rider",
            "anomaly_types": anomaly_types,
            "uncertain_flags": uncertain_flags,

            "vehicle_class": vehicle["class"],
            "vehicle_bbox": vehicle["bbox"],
            "vehicle_confidence": vehicle["confidence"],

            "rider_bbox": rider["bbox"],
            "rider_confidence": rider["confidence"],
            "rider_match_score": m["rider_score"],

            "helmet_status": helmet_result["helmet_status"],
            "helmet_confidence": helmet_result["helmet_confidence"],
            "helmet_evidence": helmet_result["evidence"],
            "head_bbox": helmet_result["head_bbox"],

            "sidewalk_status": sidewalk_result["sidewalk_status"],
            "sidewalk_confidence": sidewalk_result["sidewalk_confidence"],
            "sidewalk_evidence": sidewalk_result["evidence"],
            "vehicle_bottom_center": sidewalk_result["vehicle_bottom_center"],
        }

        # C파트 자연어 검색용 설명
        desc_parts = []
        desc_parts.append(f"{vehicle['class']} 탑승자로 추정되는 사람이 있음")
        desc_parts.append(f"탑승자 매칭 점수 {m['rider_score']}")

        if helmet_result["helmet_status"] == "no_helmet":
            desc_parts.append("헬멧 미착용으로 판정됨")
        elif helmet_result["helmet_status"] == "helmet":
            desc_parts.append("헬멧 착용으로 판정됨")
        else:
            desc_parts.append("헬멧 착용 여부는 판정 불가")

        if sidewalk_result["sidewalk_status"] == "on_sidewalk":
            desc_parts.append("인도 주행으로 판정됨")
        elif sidewalk_result["sidewalk_status"] == "not_sidewalk":
            desc_parts.append("인도 주행 아님으로 판정됨")
        else:
            desc_parts.append("인도 주행 여부는 판정 불가")

        event["description_ko"] = ". ".join(desc_parts) + "."

        events.append(event)

    frame_result = {
        "video_name": video_name,
        "frame_id": norm["frame_id"],
        "object_count": len(objects),
        "two_wheel_rider_count": len(rider_matches),
        "has_anomaly": any(len(e["anomaly_types"]) > 0 for e in events),
        "anomaly_types": sorted(set(a for e in events for a in e["anomaly_types"])),
        "uncertain_flags": sorted(set(u for e in events for u in e["uncertain_flags"])),
        "events": events,
    }

    return frame_result

In [12]:
# Cell 12. cf2 전체 처리

all_frame_results = []

for path in tqdm(json_files, desc="processing cf2 json"):
    data = load_json(path)

    video_name = data.get("video_name", path.stem)
    frames = data.get("frames", [])

    for frame in frames:
        result = judge_frame(video_name, frame)
        all_frame_results.append(result)

print("total frame results:", len(all_frame_results))

# 간단 통계
counter = Counter()
uncertain_counter = Counter()

for r in all_frame_results:
    for a in r["anomaly_types"]:
        counter[a] += 1
    for u in r["uncertain_flags"]:
        uncertain_counter[u] += 1

print("anomaly frame counts:", dict(counter))
print("uncertain frame counts:", dict(uncertain_counter))

processing cf2 json: 100%|████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 274.42it/s]

total frame results: 1835
anomaly frame counts: {}
uncertain frame counts: {'helmet_unknown': 288, 'sidewalk_unknown': 288}


In [13]:
# Cell 13. 프레임 단위 결과 저장

with open(FRAME_RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump(all_frame_results, f, ensure_ascii=False, indent=2)

print("saved:", FRAME_RESULT_PATH)

saved: C:\Users\우성한\classifier3_output\frame_anomaly_results.json


In [14]:
# Cell 14. 연속 프레임을 구간으로 묶기

def frame_sort_key(x):
    fid = x.get("frame_id")
    try:
        return int(fid)
    except:
        return str(fid)


def make_segments(frame_results, target_fields=("helmet_missing", "sidewalk_riding"), max_gap=3):
    '''
    같은 video에서 같은 anomaly_type이 max_gap 이하로 떨어져 있으면 하나의 구간으로 묶음.
    '''
    by_video = defaultdict(list)
    for r in frame_results:
        by_video[r["video_name"]].append(r)

    segments = []

    for video_name, rows in by_video.items():
        rows = sorted(rows, key=frame_sort_key)

        for anomaly_type in target_fields:
            active = []

            for r in rows:
                if anomaly_type in r["anomaly_types"]:
                    active.append(r)

            if len(active) == 0:
                continue

            current = [active[0]]

            for r in active[1:]:
                prev_id = current[-1]["frame_id"]
                curr_id = r["frame_id"]

                try:
                    gap = int(curr_id) - int(prev_id)
                except:
                    gap = 999999

                if gap <= max_gap:
                    current.append(r)
                else:
                    segments.append(build_segment(video_name, anomaly_type, current))
                    current = [r]

            if current:
                segments.append(build_segment(video_name, anomaly_type, current))

    return segments


def build_segment(video_name, anomaly_type, frames):
    frame_ids = [r["frame_id"] for r in frames]

    descriptions = []
    for r in frames[:5]:
        for e in r["events"]:
            if anomaly_type in e["anomaly_types"]:
                descriptions.append(e["description_ko"])

    return {
        "video_name": video_name,
        "anomaly_type": anomaly_type,
        "start_frame": frame_ids[0],
        "end_frame": frame_ids[-1],
        "frame_count": len(frame_ids),
        "frame_ids": frame_ids,
        "summary_ko": f"{video_name}의 {frame_ids[0]}~{frame_ids[-1]} 프레임 구간에서 {anomaly_type} 이상행동이 탐지됨.",
        "evidence_examples_ko": descriptions,
    }


segments = make_segments(all_frame_results, max_gap=3)

print("segments:", len(segments))
for s in segments[:10]:
    print(s["video_name"], s["anomaly_type"], s["start_frame"], "~", s["end_frame"], "count:", s["frame_count"])

segments: 0


In [15]:
# Cell 15. 구간 단위 결과 저장

with open(SEGMENT_RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump(segments, f, ensure_ascii=False, indent=2)

print("saved:", SEGMENT_RESULT_PATH)

saved: C:\Users\우성한\classifier3_output\segment_anomaly_results.json


In [16]:
# Cell 16. C파트 자연어 질의 테스트용 간단 검색 함수

def search_segments(query, segments):
    q = query.lower()

    keyword_map = {
        "헬멧": "helmet_missing",
        "미착용": "helmet_missing",
        "노헬멧": "helmet_missing",
        "helmet": "helmet_missing",
        "인도": "sidewalk_riding",
        "보도": "sidewalk_riding",
        "sidewalk": "sidewalk_riding",
    }

    target_types = set()
    for k, v in keyword_map.items():
        if k in q:
            target_types.add(v)

    results = []

    for s in segments:
        if len(target_types) == 0 or s["anomaly_type"] in target_types:
            results.append(s)

    return results


# 테스트
test_queries = [
    "헬멧 안 쓴 장면 찾아줘",
    "인도 주행한 구간 있어?",
]

for q in test_queries:
    print("\nQUERY:", q)
    hits = search_segments(q, segments)
    print("hits:", len(hits))
    for h in hits[:5]:
        print("-", h["video_name"], h["anomaly_type"], h["start_frame"], "~", h["end_frame"])


QUERY: 헬멧 안 쓴 장면 찾아줘
hits: 0

QUERY: 인도 주행한 구간 있어?
hits: 0


In [17]:
# Cell 17. 결과 구조 확인

if len(all_frame_results) > 0:
    print(json.dumps(all_frame_results[0], ensure_ascii=False, indent=2)[:3000])

if len(segments) > 0:
    print("\n--- segment example ---")
    print(json.dumps(segments[0], ensure_ascii=False, indent=2)[:3000])

{
  "video_name": "bs_e4387_c1.mp4",
  "frame_id": 0,
  "object_count": 0,
  "two_wheel_rider_count": 0,
  "has_anomaly": false,
  "anomaly_types": [],
  "uncertain_flags": [],
  "events": []
}


In [18]:
import os
import json
from collections import Counter

CF2_DIR = "./cf2"

counter = Counter()

for file_name in os.listdir(CF2_DIR):
    if not file_name.endswith(".json"):
        continue

    with open(os.path.join(CF2_DIR, file_name), encoding="utf-8") as f:
        data = json.load(f)

    for frame in data["frames"]:
        for cls, cnt in frame["classes"].items():
            counter[cls] += cnt

print(counter)

AttributeError: 'list' object has no attribute 'items'

In [19]:
import os
import json

CF2_DIR = "./cf2"

sample_file = os.listdir(CF2_DIR)[0]

with open(os.path.join(CF2_DIR, sample_file), encoding="utf-8") as f:
    data = json.load(f)

frame = data["frames"][0]

print(type(frame["classes"]))
print(frame["classes"])

PermissionError: [Errno 13] Permission denied: './cf2\\.ipynb_checkpoints'